# Metal Gear Agent: A Stealth-Ops Agentic AI Exercise

Welcome, operative. Your mission: build an AI **agent** (the pun — secret agent AND AI agent) to infiltrate an 8x8 military facility grid.

**Objective:** Gather **10 intel points** and survive (health > 0).

The operative starts at position (7, 0) with 3 health and 0 intel. The facility is shrouded in fog — you can only see cells adjacent to those you've visited. There are 15 total intel points available across the facility.

**Two phases:**
1. **Phase 1 (Rule-Based):** Write a `think_rule_based()` function using if/else logic, BFS navigation, and inventory tracking.
2. **Phase 2 (LLM-Powered):** Write a `think_llm()` function that uses the Gemini API to make decisions, with deterministic systems handling navigation and obvious interactions.

| TODO | Function | File | Description |
|------|----------|------|-------------|
| 1 | `think_rule_based()` | `agent.py` | Rule-based agent with BFS pathfinding |
| 2 | `think_llm()` | `agent.py` | LLM-powered agent with three-layer architecture |

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/metal_gear_agent")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from gear_agent import *
print("Metal Gear Agent loaded successfully!")

---
## Exploring the Facility

You have 9 tools available:

| Tool | Description |
|------|-------------|
| `scan()` | See adjacent cells (free action) |
| `move(direction)` | Move north/south/east/west |
| `codec(question)` | Contact NPC, safe room operator, or armory technician |
| `collect()` | Collect items on current cell |
| `equip(item)` | Fabricate/acquire equipment at armory |
| `use(item)` | Use consumable (Ration, Medkit) |
| `engage()` | Engage boss on current cell |
| `hide()` | Rest at safe room (costs 1 intel, +1 health) |
| `sitrep()` | Check operative status (free action) |

In [ ]:
# Create a game and show the full map
operative, world, tools = create_game()
tools.set_oracle(stub_oracle)
from gear_agent.display import render_grid
print(render_grid(world, operative, show_all=True))

In [ ]:
# Try out the tools manually
print("=== scan() ===")
print(tools.scan().message)
print()
print("=== move(east) ===")
print(tools.move("east").message)
print()
print("=== codec() ===")
print(tools.codec("What can you tell me about this facility?").message)
print()
print("=== sitrep() ===")
print(tools.sitrep().message)

---
## Play Interactively

Before automating anything, play the game yourself! Use the buttons to move, contact NPCs, collect items, and try to gather 10 intel.

In [ ]:
from gear_agent.interactive import play_interactive
game = play_interactive()

---
## Understanding the Tool Call Format

Your think function must return a string in this format:
```
TOOL: tool_name(arg="value")
```

Examples of valid tool calls:
```
TOOL: move(direction="north")
TOOL: collect()
TOOL: codec(question="Do you have a mission for me?")
TOOL: equip(item="EMP Device")
TOOL: engage()
TOOL: sitrep()
```

In [ ]:
# Test the parser
from gear_agent.agent import parse_tool_call

# Valid tool calls
print(parse_tool_call('TOOL: move(direction="north")'))  # ('move', {'direction': 'north'})
print(parse_tool_call('TOOL: collect()'))                  # ('collect', {})
print(parse_tool_call('TOOL: codec(question="hello")'))   # ('codec', {'question': 'hello'})

# Invalid input — falls back to scan
print(parse_tool_call('I think I should go north'))        # ('scan', {})

---
## Phase 1: Rule-Based Agent

### TODO 1: Implement `think_rule_based()`

Write a function that decides the next action using if/else logic.

**Strategy hints:**
- Use **BFS** (breadth-first search) for navigation — LLMs and if/else logic are both terrible at translating grid coordinates into cardinal directions
- Check `world.get_cell(row, col)` to see what's on the current cell
- Use `operative.visited` to track which cells you've explored
- **Priority order:** collect items > contact NPCs > fabricate equipment > engage bosses > navigate to quest targets > explore

**Available information:**
- `operative.position` — current (row, col)
- `operative.inventory` — list of item names
- `operative.has_item(name)` — check for a specific item
- `operative.health`, `operative.intel` — current stats
- `operative.visited` — set of visited cells
- `world.get_cell(r, c)` — get the Cell at (r, c)
- `world.is_passable(r, c)` — check if a cell is passable
- `cell.cell_type` — CellType enum (CORRIDOR, SERVER_ROOM, INTEL_CACHE, INFORMANT, ARMORY, SAFE_ROOM, BOSS_ARENA, etc.)
- `cell.items` — list of items on the cell
- `cell.npc_id` — NPC identifier ("the_colonel", "the_engineer", "the_informant")
- `cell.boss_name` — boss name ("Security Mainframe", "Armored Mech")
- `world.mainframe_active`, `world.mech_active` — boss status flags
- `world.errand_message_picked_up`, `world.errand_medical_picked_up` — errand status

In [ ]:
# Run Phase 1
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Intel: {result['intel']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download or view the game log
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")

---
## Phase 2: LLM-Powered Agent

### API Key Setup

In Colab, go to the **Secrets** tab (key icon in left sidebar) and add a secret named `GEMINI_API_KEY` with your Gemini API key.

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

### TODO 2: LLM Think Function

Implement `think_llm()` in the cell below. Your function should:

1. **Build a system message** with:
   - The agent's role (operative infiltrating a facility, goal is 10 intel)
   - The `TOOLS_DESCRIPTION` (imported from `gear_agent.agent`)
   - Instructions to respond with exactly one `TOOL:` call

2. **Build a user message** with:
   - Current operative status (`operative.status_text()`)
   - Recent history (last ~10 entries)
   - Current cell description
   - The operative's briefing log

3. **Call the API:**
```python
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=genai.types.GenerateContentConfig(
        system_instruction=system_message,
        max_output_tokens=500,
    ),
)
return response.text
```

### Prompt Engineering Tips

- **Ban wasted actions:** Tell the LLM "NEVER call scan() or sitrep()" — they're free but waste a turn.
- **Handle failed actions:** Include recent failures in the prompt so the LLM doesn't repeat them.
- **Encourage exploration:** When there's nothing to do, the LLM should output `TOOL: EXPLORE`.
- **Stuck detection:** If the last 3+ actions were the same, add urgency to the prompt.
- **Intel math:** Remind the LLM how much intel is still needed.

In [ ]:
from gear_agent.oracle import llm_oracle
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history, client),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Intel: {result['intel']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download or view the Phase 2 game log
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")

---
## Reflection

Compare your Phase 1 and Phase 2 agents:

1. **Intel collection:** Which agent gathered intel more efficiently? Why?
2. **NPC interactions:** How did the LLM handle codec conversations compared to hardcoded keywords?
3. **Surprises:** Did either agent discover a strategy you didn't expect?
4. **Tradeoffs:** What are the advantages of rule-based vs. LLM-powered agents?

The key insight: both agents use the same **perceive-think-act** loop. The difference is in the `think()` function — from hardcoded rules to neural network reasoning. This is the evolution from scripts to agentic AI.